# RDKit: молекулярные данные для машинного обучения

**Проект «ИИ для учёных».** Практический блокнот к разделу «Инструменты».

## Что делает этот инструмент

RDKit — стандартная библиотека хемоинформатики на Python: читает молекулы из текстового формата SMILES, вычисляет численные дескрипторы (молекулярная масса, logP, полярная поверхность и др.) и формирует молекулярные отпечатки для задач машинного обучения.

Для исследователя это значит: любой набор органических молекул можно превратить в числовую матрицу признаков и передать в стандартные модели scikit-learn. Так работают задачи QSAR (количественная связь структура–активность), предсказание растворимости, токсичности, фармакокинетических свойств.

В этом блокноте мы пройдём полный цикл: SMILES → молекула → дескрипторы → обучение модели → оценка качества. Расчётное время прохождения — около пяти минут.

## Ключевые понятия

- **SMILES** (Simplified Molecular Input Line Entry System) — текстовая запись структуры молекулы: `CCO` — этанол, `c1ccccc1` — бензол. Это стандартный формат обмена молекулярными данными в химических базах.
- **Дескриптор** — числовая характеристика молекулы, вычисляемая из её структуры: молекулярная масса, рассчитанный коэффициент распределения (logP), число атомов-доноров и акцепторов водородной связи, топологическая полярная поверхность (TPSA).
- **Молекулярный отпечаток** (fingerprint) — бинарный вектор присутствия/отсутствия подструктурных фрагментов. Отпечаток Morgan (ECFP) кодирует окрестности каждого атома и используется как признаковое представление молекулы для ML.
- **QSAR** — количественная связь структура–активность: регрессионная или классификационная модель, которая предсказывает биологическую или физико-химическую активность молекулы по её структурным признакам.

## 1. Установка библиотек

In [ ]:
!pip install -q rdkit scikit-learn pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`; вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационный набор молекул

Используем 20 распространённых органических молекул с известными свойствами. Каждая молекула задана строкой SMILES — компактной текстовой нотацией структуры. Целевая величина для QSAR-модели — синтетическая оценка растворимости (логарифм молярной растворимости log S), рассчитанная по упрощённому уравнению Есе: $\log S \approx -0.74 \cdot \log P - 0.0066 \cdot \text{MW} + 0.44$. Это игрушечная модель, полезная для демонстрации цикла предсказания свойств.

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Набор молекул: название и SMILES
MOLECULES = [
    ("этанол",          "CCO"),
    ("бензол",          "c1ccccc1"),
    ("аспирин",         "CC(=O)Oc1ccccc1C(=O)O"),
    ("кофеин",          "Cn1c(=O)c2c(ncn2C)n(c1=O)C"),
    ("глюкоза",         "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O"),
    ("ибупрофен",       "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ("парацетамол",     "CC(=O)Nc1ccc(O)cc1"),
    ("холестерин",      "CC(C)CCC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3CC=C4C[C@@H](O)CC[C@]4(C)[C@H]3CC[C@]12C"),
    ("нафталин",        "c1ccc2ccccc2c1"),
    ("аденин",          "Nc1ncnc2ncnc12"),
    ("этилацетат",      "CCOC(=O)C"),
    ("ацетон",          "CC(C)=O"),
    ("уксусная кислота","CC(=O)O"),
    ("толуол",          "Cc1ccccc1"),
    ("анилин",          "Nc1ccccc1"),
    ("фенол",           "Oc1ccccc1"),
    ("глицин",          "NCC(=O)O"),
    ("хлорамфеникол",   "ClC(Cl)C(=O)N[C@@H](CO)[C@H](O)c1ccc([N+](=O)[O-])cc1"),
    ("никотин",         "CN1CCC[C@H]1c1cccnc1"),
    ("капсаицин",       "COc1cc(CNC(=O)CCCC/C=C/C(C)C)ccc1O"),
]

# Парсим SMILES → объекты Mol
records = []
for name, smi in MOLECULES:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        mw  = Descriptors.MolWt(mol)
        lp  = Descriptors.MolLogP(mol)
        records.append({"название": name, "SMILES": smi, "MW": mw, "logP": lp})

df_mols = pd.DataFrame(records)
print(f"Успешно разобрано молекул: {len(df_mols)}")
df_mols.round(2)

## 4. Применение инструмента

### Шаг 4.1. Отрисовка структур молекул

RDKit умеет генерировать структурные формулы. Ячейка ниже строит сетку изображений для первых восьми молекул. В реальных задачах это удобно для быстрого просмотра набора и проверки корректности SMILES.

In [ ]:
from rdkit.Chem import Draw
from IPython.display import display

# Объекты Mol для первых восьми молекул
mols_display = [Chem.MolFromSmiles(smi) for _, smi in MOLECULES[:8]]
labels       = [name for name, _ in MOLECULES[:8]]

# MolsToGridImage возвращает PIL.Image; display выводит его в Colab
img = Draw.MolsToGridImage(
    mols_display,
    molsPerRow=4,
    subImgSize=(300, 220),
    legends=labels,
)
display(img)

### Шаг 4.2. Расчёт молекулярных дескрипторов

Для каждой молекулы вычислим шесть физико-химических дескрипторов, которые широко используются в QSAR и drug discovery:

| Дескриптор | Смысл |
|---|---|
| **MW** | Молекулярная масса (г/моль) |
| **logP** | Коэффициент распределения октанол/вода (гидрофобность) |
| **HBD** | Число доноров водородной связи |
| **HBA** | Число акцепторов водородной связи |
| **TPSA** | Топологическая полярная поверхность (Å²) |
| **RotBonds** | Число вращаемых связей (гибкость молекулы) |

Правило пяти Липински (критерий «лекарственности» молекулы): MW < 500, logP < 5, HBD ≤ 5, HBA ≤ 10.

In [ ]:
from rdkit.Chem import rdMolDescriptors

desc_records = []
for name, smi in MOLECULES:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    desc_records.append({
        "название":  name,
        "MW":        round(Descriptors.MolWt(mol), 2),
        "logP":      round(Descriptors.MolLogP(mol), 2),
        "HBD":       rdMolDescriptors.CalcNumHBD(mol),
        "HBA":       rdMolDescriptors.CalcNumHBA(mol),
        "TPSA":      round(rdMolDescriptors.CalcTPSA(mol), 1),
        "RotBonds":  rdMolDescriptors.CalcNumRotatableBonds(mol),
    })

df_desc = pd.DataFrame(desc_records)
# Правило Липински: молекулы, удовлетворяющие всем критериям
df_desc["Липински_ОК"] = (
    (df_desc["MW"] < 500) &
    (df_desc["logP"] < 5) &
    (df_desc["HBD"] <= 5) &
    (df_desc["HBA"] <= 10)
)
print(f"Молекул, удовлетворяющих правилу Липински: "
      f"{df_desc['Липински_ОК'].sum()} из {len(df_desc)}")
df_desc

### Шаг 4.3. Визуализация пространства химических свойств

Построим диаграмму рассеяния MW–logP, стандартную в medicinal chemistry для оценки «лекарственного пространства». Цвет кодирует соответствие правилу Липински.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5.5))

colors = [VIZ["series"][0] if ok else VIZ["series"][2]
          for ok in df_desc["Липински_ОК"]]

ax.scatter(df_desc["MW"], df_desc["logP"], c=colors, s=80, zorder=3)

# Подписи для каждой молекулы
for _, row in df_desc.iterrows():
    ax.annotate(
        row["название"],
        xy=(row["MW"], row["logP"]),
        xytext=(4, 4), textcoords="offset points",
        fontsize=8, color=VIZ["node_text"],
    )

# Границы «лекарственного пространства» Липински
ax.axvline(500, color=VIZ["edge"], linestyle="--", linewidth=1.2, alpha=0.7,
           label="Липински: MW = 500")
ax.axhline(5,   color=VIZ["grid"],  linestyle="--", linewidth=1.2, alpha=0.9,
           label="Липински: logP = 5")

patch_ok  = mpatches.Patch(color=VIZ["series"][0], label="Правило Липински выполнено")
patch_no  = mpatches.Patch(color=VIZ["series"][2], label="Нарушает правило Липински")
ax.legend(handles=[patch_ok, patch_no], loc="upper left", fontsize=10)

ax.set_title("Химическое пространство набора молекул")
ax.set_xlabel("Молекулярная масса MW (г/моль)")
ax.set_ylabel("logP (гидрофобность)")
fig.tight_layout()
plt.show()

### Шаг 4.4. Молекулярные отпечатки Morgan (ECFP)

Отпечатки Morgan — наиболее распространённое признаковое представление молекул для ML. Каждый бит вектора отвечает за наличие определённого кольцевого фрагмента в молекуле. Параметр `radius=2` соответствует ECFP4 (учитывает окрестность радиуса 2 вокруг каждого атома); `nBits=1024` — длина вектора.

In [ ]:
from rdkit.Chem import AllChem
import numpy as np

def morgan_fingerprint(smi, radius=2, n_bits=1024):
    """Вычисляет отпечаток Morgan для строки SMILES."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    return np.array(fp)

# Матрица отпечатков: строки — молекулы, столбцы — биты вектора
fp_matrix = np.array([morgan_fingerprint(smi) for _, smi in MOLECULES])
print(f"Матрица отпечатков: {fp_matrix.shape}  "
      f"(молекул × битов)")
print(f"Ненулевых битов у аспирина: {fp_matrix[2].sum()}")

# Сходство Танимото между первыми двумя молекулами (этанол и бензол)
def tanimoto(a, b):
    """Коэффициент сходства Танимото для бинарных векторов."""
    intersection = np.dot(a, b)
    return intersection / (a.sum() + b.sum() - intersection)

print(f"\nСходство Танимото: этанол–бензол = {tanimoto(fp_matrix[0], fp_matrix[1]):.3f}")
print(f"Сходство Танимото: аспирин–парацетамол = {tanimoto(fp_matrix[2], fp_matrix[6]):.3f}")

### Шаг 4.5. Мини-пример QSAR: предсказание растворимости

Обучим модель RandomForestRegressor предсказывать синтетическую растворимость (log S) по дескрипторам молекул. Целевая величина вычислена по упрощённой формуле Есе — это позволяет демонстрировать полный ML-цикл без использования приватных экспериментальных данных. Признаки — шесть дескрипторов RDKit.

Важно: выборка из 20 молекул мала для реального исследования. Здесь задача — показать цикл, а не достичь высокого качества. Для настоящей QSAR-модели используйте датасеты из открытых репозиториев (ChEMBL, PubChem, ESOL).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Признаки: шесть дескрипторов
feature_cols = ["MW", "logP", "HBD", "HBA", "TPSA", "RotBonds"]
X = df_desc[feature_cols].values.astype(float)

# Целевая величина: log S по упрощённой формуле Есе
# logS ≈ -0.74*logP - 0.0066*MW + 0.44
y = -0.74 * df_desc["logP"].values - 0.0066 * df_desc["MW"].values + 0.44

print("Целевые значения log S (синтетические):")
for name, val in zip(df_desc["название"], y):
    print(f"  {name:<22} {val:+.2f}")

# Кросс-валидация leave-one-out (LOO): при малой выборке — стандартная практика
rf = RandomForestRegressor(n_estimators=200, random_state=42)
loo = LeaveOneOut()
loo_scores = cross_val_score(rf, X, y, cv=loo, scoring="r2")
loo_mae    = -cross_val_score(rf, X, y, cv=loo, scoring="neg_mean_absolute_error")

print(f"\nCross-val LOO  R² = {loo_scores.mean():.3f} ± {loo_scores.std():.3f}")
print(f"Cross-val LOO MAE = {loo_mae.mean():.3f}")

In [ ]:
import matplotlib.pyplot as plt

# Обучаем на всей выборке для визуализации важности признаков
rf.fit(X, y)
importances = rf.feature_importances_
order = np.argsort(importances)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Важность признаков
axes[0].barh(
    np.array(feature_cols)[order],
    importances[order],
    color=VIZ["series"][1],
)
axes[0].set_title("Важность признаков в QSAR-модели")
axes[0].set_xlabel("Важность (Gini)")

# Предсказание (обучение) против истинных значений log S
y_pred_train = rf.predict(X)
axes[1].scatter(y, y_pred_train, color=VIZ["series"][0], s=60, zorder=3)
lims = [y.min() - 0.2, y.max() + 0.2]
axes[1].plot(lims, lims, color=VIZ["series"][2], linestyle="--",
             label="Идеальное совпадение")
for i, name in enumerate(df_desc["название"]):
    axes[1].annotate(name, (y[i], y_pred_train[i]), fontsize=7,
                     xytext=(3, 3), textcoords="offset points",
                     color=VIZ["node_text"])
axes[1].set_title("Прогноз vs. истина (обучающая выборка)")
axes[1].set_xlabel("Истинный log S")
axes[1].set_ylabel("Предсказанный log S")
axes[1].legend()

fig.tight_layout()
plt.show()
print("Напоминание: LOO R² — более честная оценка на этой малой выборке.")

## 5. Интерпретация результатов

- **Диаграмма MW–logP**: молекулы в левом нижнем углу (низкая масса, низкая гидрофобность) обычно хорошо растворимы в воде. Правило Липински отсекает большинство нелекарственноподобных соединений.
- **Коэффициент сходства Танимото**: значение 0 — полностью разные отпечатки, 1 — идентичные. Аспирин и парацетамол имеют ненулевое сходство, потому что оба содержат бензольное кольцо с заместителями.
- **Важность признаков**: logP и MW доминируют, потому что именно из них аналитически вычислена целевая величина. На реальных данных картина будет другой и информативной.
- **LOO R²** — честная оценка для малой выборки: модель предсказывает каждую молекулу, обучившись на остальных. При низком LOO R² нужен более крупный датасет или более богатые признаки (например, 2048-битные отпечатки Morgan вместо 6 дескрипторов).
- На реальных наборах (~1000+ молекул) отпечатки Morgan дают точность выше, чем дескрипторы, за счёт большего информационного содержания.

## 6. Попробуйте на своих данных

Замените список `MOLECULES` своим набором SMILES. Целевые значения добавьте в список `y_custom`.

1. Экспортируйте ваши молекулы в формате SMILES (ChEMBL, PubChem, MOL → SMILES через RDKit).
2. Заполните ячейку ниже своими данными.
3. Запустите ячейки раздела 4 заново — дескрипторы и модель пересчитаются автоматически.
4. Для работы с отпечатками Morgan вместо дескрипторов замените матрицу `X` на `fp_matrix` из шага 4.4.

In [ ]:
# --- Шаблон для своих данных ---
# MY_MOLECULES = [
#     ("название_1", "SMILES_строка_1"),
#     ("название_2", "SMILES_строка_2"),
#     # ... добавьте свои молекулы
# ]
# y_custom = [значение_1, значение_2, ...]   # экспериментальные значения
#
# Замените MOLECULES = MY_MOLECULES в ячейке раздела 3 и y = np.array(y_custom)
# в ячейке раздела 4.5, затем выполните блокнот заново.
print("Шаблон готов. Заполните MY_MOLECULES и y_custom, затем перезапустите блокнот.")

## Что дальше

- Официальная документация RDKit: https://www.rdkit.org/docs/
- Книга «Getting Started with the RDKit in Python»: https://www.rdkit.org/docs/GettingStartedInPython.html
- Открытый датасет ESOL (1128 молекул, экспериментальная растворимость): Delaney J.S., J. Chem. Inf. Comput. Sci., 2004.
- Для крупных наборов молекул используйте базу ChEMBL (chembl.ebi.ac.uk) или PubChem.
- Продвинутые представления молекул для ML: графовые нейронные сети (GNN) через библиотеку DGL-LifeSci или PyTorch Geometric.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы получили SMILES-строку из внешней базы, рассчитали дескрипторы и обнаружили, что `Chem.MolFromSmiles()` вернула `None` для нескольких молекул. Что это означает и как правильно обработать такой результат перед обучением QSAR-модели?
2. Модель, обученная на дескрипторах шести физико-химических свойств, показала высокий R² на обучающей выборке, но низкий LOO R² на тех же 20 молекулах. В чём наиболее вероятная причина и как изменить признаковое представление, чтобы улучшить обобщение?
3. Два соединения имеют сходство Танимото по отпечаткам Morgan (ECFP4, radius=2) равное 0.85. Означает ли это, что они обладают схожей биологической активностью, и какие аспекты молекулярного строения отпечаток ECFP4 не кодирует?

<details>
<summary>Показать ориентиры для ответов</summary>

1. `None` означает, что SMILES не удалось разобрать: строка содержит синтаксическую ошибку, недопустимую валентность или некорректную стереохимическую нотацию. Такие молекулы необходимо исключить из датасета до расчёта дескрипторов; нельзя молча подставлять нулевые значения — это внесёт систематическую ошибку в модель.
2. Переобучение на малой выборке: шесть дескрипторов почти однозначно восстанавливают синтетическую целевую переменную, вычисленную из части тех же дескрипторов (logP, MW). При использовании реальных экспериментальных значений активности и более богатых признаков (например, 1024-битных отпечатков Morgan) LOO R² будет отражать подлинную прогностическую способность. Увеличение датасета до сотен молекул также снизит дисперсию оценки.
3. Высокое сходство Танимото указывает на структурное сходство фрагментов в ближайшей окрестности атомов, однако не гарантирует похожей активности: разные заместители на периферии молекулы, другая стереохимия или иное расположение фармакофорных групп могут кардинально изменить биологический эффект. Кроме того, ECFP4 не кодирует 3D-конформацию, торсионные углы и межмолекулярные взаимодействия с рецептором.
</details>